# Implementasi Webinar: Make Sense of Data with Analytics and AI

## Studi Kasus Baru: Analisis dan Prediksi Permintaan Produk di PT SegarMart

**Tema:** Data Analytics, Machine Learning, dan dukungan Generative AI  
**Platform:** Python — Google Colab / Jupyter Notebook  
**Jenis data:** Dataset simulasi

### Latar belakang
PT SegarMart merupakan bisnis ritel yang ingin memahami pola penjualan dan memperkirakan permintaan produk untuk membantu perencanaan persediaan. Perusahaan menghadapi tantangan berupa perubahan permintaan berdasarkan waktu, harga, promosi, dan jenis produk.

Notebook ini menerapkan alur analitik yang sama dengan materi webinar, tetapi menggunakan permasalahan yang berbeda dari analisis keterlambatan pembayaran faktur. Tahapan mencakup pemahaman data, pembersihan, *descriptive analytics*, *diagnostic analytics*, prediksi permintaan, evaluasi model, dan rekomendasi bisnis.

**Catatan:** seluruh data dibuat secara sintetis untuk pembelajaran dan tidak mewakili data perusahaan nyata.

## 1. Persiapan pustaka

Pustaka `pandas` dan `numpy` digunakan untuk pengolahan data, `matplotlib` dan `seaborn` untuk visualisasi, serta `scikit-learn` untuk membangun dan mengevaluasi model prediksi.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Pustaka berhasil dimuat.")

## 2. Membuat dataset simulasi

Dataset berisi catatan penjualan harian untuk beberapa produk selama dua tahun. Setiap baris mewakili satu produk pada satu tanggal.

Variabel utama:
- `date`: tanggal transaksi.
- `product`: nama produk.
- `category`: kategori produk.
- `price`: harga jual per unit.
- `promo`: penanda apakah produk sedang dipromosikan.
- `day_of_week`, `month`, `is_weekend`: fitur kalender.
- `units_sold`: jumlah unit yang terjual, sebagai target prediksi.

Nilai permintaan dibuat dengan pola musiman dan pengaruh promosi serta harga agar latihan analisis memiliki pola yang dapat dipelajari.

In [ ]:
dates = pd.date_range("2024-01-01", "2025-12-31", freq="D")
product_info = pd.DataFrame({
    "product": ["Beras 5 kg", "Minyak Goreng", "Gula 1 kg", "Susu UHT", "Mi Instan", "Kopi", "Telur", "Tepung"],
    "category": ["Bahan Pokok", "Bahan Pokok", "Bahan Pokok", "Minuman", "Makanan Instan", "Minuman", "Segar", "Bahan Pokok"],
    "base_price": [72000, 18000, 16500, 19000, 3500, 22000, 28000, 14000],
    "base_demand": [28, 42, 25, 34, 110, 22, 38, 20]
})

rows = []
for _, p in product_info.iterrows():
    for date in dates:
        dow = date.dayofweek
        month = date.month
        is_weekend = int(dow >= 5)
        promo = int(rng.random() < (0.24 if is_weekend else 0.15))
        price_factor = rng.normal(1.0, 0.035) - (0.08 if promo else 0)
        price = max(1000, p["base_price"] * price_factor)
        seasonal = 1 + 0.12 * np.sin(2 * np.pi * date.dayofyear / 365.25)
        weekend_effect = 1.18 if is_weekend else 1.0
        promo_effect = 1.35 if promo else 1.0
        price_effect = (p["base_price"] / price) ** 0.7
        trend = 1 + 0.00025 * (date - dates[0]).days
        expected = p["base_demand"] * seasonal * weekend_effect * promo_effect * price_effect * trend
        units = max(0, int(rng.poisson(max(expected, 0.1))))
        rows.append((date, p["product"], p["category"], round(price, 2), promo, units))

sales = pd.DataFrame(rows, columns=["date", "product", "category", "price", "promo", "units_sold"])
sales["day_of_week"] = sales["date"].dt.dayofweek
sales["month"] = sales["date"].dt.month
sales["is_weekend"] = (sales["day_of_week"] >= 5).astype(int)

# Tambahkan sedikit masalah kualitas data untuk latihan pembersihan.
sales_raw = sales.copy()
missing_idx = sales_raw.sample(frac=0.005, random_state=SEED).index
sales_raw.loc[missing_idx, "price"] = np.nan
sales_raw = pd.concat([sales_raw, sales_raw.sample(8, random_state=SEED)], ignore_index=True)

print("Ukuran data mentah:", sales_raw.shape)
display(sales_raw.head())
display(product_info)

## 3. Memahami struktur dan kualitas data

Tahap *Data Understanding* memeriksa contoh baris, tipe data, nilai kosong, duplikasi, serta statistik dasar. Pemeriksaan ini membantu mengetahui apakah data layak digunakan sebelum analisis lebih lanjut.

In [ ]:
display(sales_raw.info())
display(sales_raw.describe(include="all").T)

quality_report = pd.DataFrame({
    "kolom": sales_raw.columns,
    "tipe_data": [str(sales_raw[c].dtype) for c in sales_raw.columns],
    "nilai_kosong": [sales_raw[c].isna().sum() for c in sales_raw.columns],
    "jumlah_unik": [sales_raw[c].nunique(dropna=True) for c in sales_raw.columns]
})
display(quality_report)
print("Jumlah baris duplikat penuh:", sales_raw.duplicated().sum())

## 4. Membersihkan dan menyiapkan data

Baris duplikat dihapus. Nilai harga yang kosong diisi menggunakan median harga produk yang sama, agar pengisian tetap mempertimbangkan perbedaan harga ant produk. Setelah itu, tipe tanggal dan nilai yang tidak masuk akal diperiksa.

In [ ]:
sales_clean = sales_raw.drop_duplicates().copy()
sales_clean["date"] = pd.to_datetime(sales_clean["date"], errors="coerce")

# Isi harga kosong menggunakan median untuk produk yang sama.
sales_clean["price"] = sales_clean.groupby("product")["price"].transform(
    lambda s: s.fillna(s.median())
)

sales_clean = sales_clean.dropna(subset=["date", "product", "units_sold"])
sales_clean = sales_clean[(sales_clean["price"] > 0) & (sales_clean["units_sold"] >= 0)]

print("Baris sebelum dibersihkan:", len(sales_raw))
print("Baris setelah dibersihkan:", len(sales_clean))
print("Duplikasi tersisa:", sales_clean.duplicated().sum())
print("Nilai kosong tersisa:")
display(sales_clean.isna().sum().to_frame("jumlah_kosong"))

## 5. Descriptive Analytics — Apa yang terjadi?

Analitik deskriptif merangkum performa penjualan: total unit terjual, pendapatan estimasi, produk terlaris, serta pola penjualan dari bulan ke bulan. Pendapatan estimasi dihitung sebagai harga dikali unit terjual dan bukan laporan keuangan resmi.

In [ ]:
sales_clean["revenue"] = sales_clean["price"] * sales_clean["units_sold"]
monthly_sales = (
    sales_clean.assign(month_start=sales_clean["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month_start", as_index=False)
    .agg(units_sold=("units_sold", "sum"), revenue=("revenue", "sum"))
)

product_summary = (
    sales_clean.groupby(["product", "category"], as_index=False)
    .agg(total_units=("units_sold", "sum"), estimated_revenue=("revenue", "sum"))
    .sort_values("total_units", ascending=False)
)

print("Total unit terjual:", int(sales_clean["units_sold"].sum()))
print(f"Estimasi pendapatan: Rp{sales_clean['revenue'].sum():,.0f}")
display(product_summary)
display(monthly_sales.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))
axes[0].plot(monthly_sales["month_start"], monthly_sales["units_sold"], marker="o")
axes[0].set_title("Total Unit Terjual per Bulan")
axes[0].set_ylabel("Unit")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(product_summary["product"], product_summary["total_units"])
axes[1].set_title("Total Unit Terjual menurut Produk")
axes[1].set_ylabel("Unit")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 6. Diagnostic Analytics — Faktor apa yang berkaitan dengan permintaan?

Analisis diagnostik membandingkan penjualan berdasarkan kategori, hari kerja versus akhir pekan, dan status promosi. Perbedaan ini dapat memberi petunjuk untuk ditelusuri, tetapi tidak otomatis membuktikan hubungan sebab-akibat.

In [ ]:
sales_clean["period_type"] = np.where(sales_clean["is_weekend"] == 1, "Akhir pekan", "Hari kerja")
by_period = sales_clean.groupby("period_type", as_index=False).agg(
    average_units=("units_sold", "mean"),
    total_units=("units_sold", "sum")
)
by_promo = sales_clean.groupby("promo", as_index=False).agg(
    average_units=("units_sold", "mean"),
    transaction_days=("units_sold", "count")
)
by_category = sales_clean.groupby("category", as_index=False).agg(
    average_units=("units_sold", "mean"),
    total_units=("units_sold", "sum")
)

display(by_period)
display(by_promo)
display(by_category)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(by_period["period_type"], by_period["average_units"])
axes[0].set_title("Rata-rata Penjualan: Hari Kerja vs Akhir Pekan")
axes[0].set_xlabel("")
axes[0].set_ylabel("Rata-rata unit")
axes[1].bar(by_promo["promo"].astype(str), by_promo["average_units"])
axes[1].set_title("Rata-rata Penjualan berdasarkan Promosi")
axes[1].set_xlabel("Promosi (0 = tidak, 1 = ya)")
axes[1].set_ylabel("Rata-rata unit")
plt.tight_layout()
plt.show()

## 7. Predictive Analytics — Memprediksi jumlah permintaan produk

Target model adalah `units_sold`, yaitu jumlah unit terjual pada satu hari untuk suatu produk. Fitur meliputi produk, kategori, harga, promosi, dan informasi kalender. Data dibagi berdasarkan waktu: periode awal untuk pelatihan dan periode akhir untuk pengujian.

**Catatan:** model ini merupakan demonstrasi pembelajaran. Dalam penerapan nyata, fitur seperti promosi dan harga hanya boleh digunakan jika nilainya sudah diketahui pada saat prediksi dibuat.

In [ ]:
model_df = sales_clean.copy().sort_values("date").reset_index(drop=True)
feature_cols = ["product", "category", "price", "promo", "day_of_week", "month", "is_weekend"]
target_col = "units_sold"

cutoff = model_df["date"].quantile(0.80)
train_df = model_df[model_df["date"] < cutoff].copy()
test_df = model_df[model_df["date"] >= cutoff].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print("Batas waktu pemisahan:", cutoff.date())
print("Jumlah data latih:", len(train_df))
print("Jumlah data uji:", len(test_df))

In [ ]:
numeric_features = ["price", "promo", "day_of_week", "month", "is_weekend"]
categorical_features = ["product", "category"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=160, max_depth=12, min_samples_leaf=3, random_state=SEED, n_jobs=-1
    )
}

results = []
fitted_models = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = np.maximum(pipe.predict(X_test), 0)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("MAE")
display(results_df)

## 8. Evaluasi model

Model regresi dievaluasi menggunakan:
- **MAE:** rata-rata selisih absolut antara prediksi dan nilai aktual; semakin kecil semakin baik.
- **RMSE:** memberi penalti lebih besar pada kesalahan yang besar; semakin kecil semakin baik.
- **R²:** mengukur seberapa besar variasi target yang dapat dijelaskan model; nilai mendekati 1 umumnya lebih baik.

Metrik perlu dibaca bersama konteks bisnis. Kesalahan prediksi beberapa unit mungkin kecil untuk produk berpermintaan tinggi, tetapi besar untuk produk berpermintaan rendah.

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
predictions = np.maximum(best_model.predict(X_test), 0)

comparison = test_df[["date", "product", "units_sold"]].copy()
comparison["predicted_units"] = np.round(predictions, 1)
comparison["absolute_error"] = np.abs(comparison["units_sold"] - comparison["predicted_units"])
print("Model terbaik berdasarkan MAE:", best_name)
display(comparison.head(15))

plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions, alpha=0.35)
lims = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Penjualan aktual (unit)")
plt.ylabel("Prediksi penjualan (unit)")
plt.title(f"Aktual vs Prediksi — {best_name}")
plt.tight_layout()
plt.show()

## 9. Prescriptive Analytics — Rekomendasi persediaan

Hasil analisis dapat digunakan sebagai bahan pertimbangan untuk perencanaan stok. Contoh berikut menghitung rata-rata permintaan dan menyusun perkiraan stok sederhana berdasarkan rata-rata permintaan harian pada periode terbaru.

Rekomendasi ini bukan keputusan otomatis. Stok pengaman, waktu pengadaan, masa simpan, dan kapasitas gudang perlu dipertimbangkan sebelum diterapkan.

In [ ]:
recent_cutoff = sales_clean["date"].max() - pd.Timedelta(days=30)
recent = sales_clean[sales_clean["date"] > recent_cutoff]

inventory_plan = recent.groupby(["product", "category"], as_index=False).agg(
    avg_daily_units=("units_sold", "mean"),
    daily_std=("units_sold", "std")
)
inventory_plan["suggested_7day_stock"] = np.ceil(
    inventory_plan["avg_daily_units"] * 7 + 1.65 * inventory_plan["daily_std"].fillna(0) * np.sqrt(7)
).astype(int)
inventory_plan = inventory_plan.sort_values("suggested_7day_stock", ascending=False)
display(inventory_plan)

print("Rekomendasi awal:")
print("- Prioritaskan pemantauan stok produk dengan perkiraan kebutuhan tujuh hari tertinggi.")
print("- Tinjau pola akhir pekan dan promosi saat merencanakan persediaan.")
print("- Perbarui prediksi ketika harga, promosi, atau pola permintaan berubah.")
print("- Bandingkan prediksi dengan penjualan aktual secara berkala sebelum digunakan untuk keputusan pengadaan.")

## 10. Peran Generative AI dalam workflow

Generative AI dapat membantu menjelaskan kode, mengusulkan pemeriksaan kualitas data, atau merangkum hasil analisis. Hasil AI tetap harus diverifikasi dan tidak menggantikan penilaian manusia.

Contoh prompt:
> “Berdasarkan metrik MAE, RMSE, dan R² dari model prediksi permintaan, jelaskan model mana yang lebih sesuai untuk perencanaan persediaan. Sebutkan keterbatasan evaluasi dan pemeriksaan lanjutan yang perlu dilakukan. Jangan mengarang nilai yang tidak tersedia pada hasil analisis.”

## 11. Kesimpulan

Implementasi ini menunjukkan penggunaan data analytics dan machine learning untuk membantu perencanaan permintaan ritel. Analitik deskriptif merangkum penjualan, analitik diagnostik membandingkan pola menurut waktu dan promosi, sedangkan model regresi memperkirakan jumlah unit terjual. Hasil prediksi kemudian diterjemahkan menjadi rekomendasi awal untuk pengelolaan persediaan.

Seluruh data dalam notebook merupakan data sintetis. Oleh karena itu, hasil tidak boleh dianggap sebagai gambaran bisnis nyata. Sebelum diterapkan, model perlu diuji dengan data operasional yang relevan, dibandingkan dengan metode dasar, dan dipantau secara berkala.

### Keterbatasan
- Dataset dibuat untuk pembelajaran dan tidak mewakili transaksi perusahaan nyata.
- Variabel seperti hari libur, cuaca, ketersediaan stok, dan aktivitas pesaing belum dimasukkan.
- Pembagian waktu sederhana belum menggantikan validasi deret waktu yang lebih menyeluruh.
- Rekomendasi stok belum memasukkan waktu tunggu pemasok, biaya penyimpanan, dan masa kedaluwarsa.

### Keterkaitan dengan materi webinar
Notebook menerapkan tahapan pemahaman data, persiapan data, analisis deskriptif dan diagnostik, pemodelan prediktif, evaluasi, serta penyusunan rekomendasi. Generative AI ditempatkan sebagai alat bantu analisis yang tetap memerlukan verifikasi manusia.